# Stage 3: Data Preprocessing & Cleaning

This notebook handles all preprocessing steps: resizing, normalization, augmentation, and dataset splitting.

In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

DATA_ROOT = Path('../data/mvtec_anomaly_detection')
CATEGORY = 'bottle'
IMG_SIZE = 224
BATCH_SIZE = 32

# ImageNet stats for normalization
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

print('Config set.')
print(f'Image size: {IMG_SIZE}x{IMG_SIZE}')
print(f'Batch size: {BATCH_SIZE}')
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')

In [ ]:
# ============================================================
# Transforms
# ============================================================

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

print('Transforms defined.')
print('Train: resize → random flip/rotate → color jitter → normalize')
print('Val/Test: resize → normalize')

In [ ]:
class MVTecDataset(Dataset):
    """
    PyTorch Dataset for MVTec Anomaly Detection.
    
    Args:
        root: Path to mvtec_anomaly_detection/
        category: e.g. 'bottle'
        split: 'train' or 'test'
        transform: torchvision transforms
    """
    def __init__(self, root, category, split='train', transform=None):
        self.root = Path(root)
        self.category = category
        self.split = split
        self.transform = transform
        
        self.image_paths = []
        self.labels = []       # 0=normal, 1=defective
        self.defect_types = []
        
        split_path = self.root / category / split
        
        for defect_dir in sorted(split_path.iterdir()):
            if not defect_dir.is_dir():
                continue
            label = 0 if defect_dir.name == 'good' else 1
            for img_path in sorted(defect_dir.glob('*.png')):
                self.image_paths.append(img_path)
                self.labels.append(label)
                self.defect_types.append(defect_dir.name)
        
        print(f'[{split}] Loaded {len(self.image_paths)} images '
              f'({sum(l==0 for l in self.labels)} normal, '
              f'{sum(l==1 for l in self.labels)} defective)')
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return img, label


# ============================================================
# Instantiate datasets (requires downloaded data)
# ============================================================
if DATA_ROOT.exists():
    train_dataset = MVTecDataset(DATA_ROOT, CATEGORY, split='train', transform=train_transforms)
    test_dataset  = MVTecDataset(DATA_ROOT, CATEGORY, split='test',  transform=val_transforms)
    
    # Split test into val + test
    val_size = int(0.5 * len(test_dataset))
    test_size = len(test_dataset) - val_size
    val_dataset, test_dataset_final = torch.utils.data.random_split(
        test_dataset, [val_size, test_size]
    )
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
    val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    test_loader  = DataLoader(test_dataset_final, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    print(f'\nDataLoaders ready.')
    print(f'Train batches: {len(train_loader)}')
    print(f'Val batches:   {len(val_loader)}')
    print(f'Test batches:  {len(test_loader)}')
else:
    print('Dataset not found — download MVTec AD first.')

In [ ]:
# ============================================================
# Verify normalization by checking pixel stats
# ============================================================
def check_normalization(loader, n_batches=5):
    means, stds = [], []
    for i, (imgs, _) in enumerate(loader):
        if i >= n_batches:
            break
        means.append(imgs.mean(dim=[0,2,3]))
        stds.append(imgs.std(dim=[0,2,3]))
    print(f'Mean after normalization: {torch.stack(means).mean(0).numpy().round(3)}')
    print(f'Std  after normalization: {torch.stack(stds).mean(0).numpy().round(3)}')
    print('(Should be close to [0,0,0] mean and [1,1,1] std)')

if DATA_ROOT.exists():
    check_normalization(train_loader)

## 3.1 Handling Class Imbalance

Since the training set contains **only normal images** (MVTec standard), we handle class imbalance during testing as follows:
- Use **weighted loss** (higher weight for defective class) during supervised training
- Use **AUROC** as primary metric (not accuracy)
- Optionally use **oversampling** of defective test samples for validation

In [ ]:
# Compute class weights for weighted CrossEntropy
# class_counts = [n_normal, n_defective]
# weights[i] = total / (n_classes * class_counts[i])

def compute_class_weights(labels):
    labels = np.array(labels)
    counts = np.bincount(labels)
    total = len(labels)
    weights = total / (len(counts) * counts)
    print(f'Class counts: Normal={counts[0]}, Defective={counts[1]}')
    print(f'Class weights: Normal={weights[0]:.3f}, Defective={weights[1]:.3f}')
    return torch.FloatTensor(weights)

if DATA_ROOT.exists():
    class_weights = compute_class_weights(test_dataset.labels)
else:
    # Demo
    demo_labels = [0]*83 + [1]*62
    class_weights = compute_class_weights(demo_labels)